In [1]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [3]:
# MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

| Parameter                          | Simple Meaning                        | Usually Change?      |
| ---------------------------------- | ------------------------------------- | -------------------- |
| `load_in_8bit`                     | Load model in INT8                    | ✅ Yes                |
| `load_in_4bit`                     | Load model in 4-bit                   | ✅ Yes                |
| `llm_int8_threshold`               | Outlier cutoff for FP16 vs INT8       | ❌ No                 |
| `llm_int8_skip_modules`            | Layers to keep unquantized            | Sometimes            |
| `llm_int8_enable_fp32_cpu_offload` | Move some layers to CPU               | Sometimes            |
| `llm_int8_has_fp16_weight`         | Keep FP16 copy for training           | Training only        |
| `bnb_4bit_compute_dtype`           | Precision used for computation        | ✅ Often (`bfloat16`) |
| `bnb_4bit_quant_type`              | Choose FP4 or NF4                     | ✅ Usually `nf4`      |
| `bnb_4bit_use_double_quant`        | Quantize scaling constants too        | ✅ Often for QLoRA    |
| `bnb_4bit_quant_storage`           | How 4-bit values are packed in memory | ❌ Rarely             |


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [6]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

| Parameter           | What it controls               | Recommended                                                         |
| ------------------- | ------------------------------ | ------------------------------------------------------------------- |
| `r`                 | Adapter size (capacity)        | 8–32                                                                |
| `target_modules`    | Which layers get LoRA          | `q_proj`, `k_proj`, `v_proj`, `o_proj` (often also MLP projections) |
| `lora_alpha`        | Strength of the LoRA update    | `2 × r` is a common starting point                                  |
| `lora_dropout`      | Regularization during training | `0.05`                                                              |
| `bias`              | Whether to train bias terms    | `"none"`                                                            |
| `modules_to_save`   | Extra layers to save           | Usually `None`                                                      |
| `use_rslora`        | More stable scaling            | `True` for many modern setups                                       |
| `init_lora_weights` | Adapter initialization         | `True` (default)                                                    |


In [15]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj","up_proj","down_proj"
],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [17]:
lora_adapter_model = get_peft_model(base_model, lora_config)

In [18]:
lora_adapter_model.print_trainable_parameters()

trainable params: 6,307,840 || all params: 1,106,356,224 || trainable%: 0.5701


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338

trainable params: 6,307,840 || all params: 1,106,356,224 || trainable%: 0.5701

In [19]:
total_params = lora_adapter_model.num_parameters()
trainable_params = lora_adapter_model.num_parameters(only_trainable=True)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable %:", 100 * trainable_params / total_params)

Total parameters: 1106356224
Trainable parameters: 6307840
Trainable %: 0.570145479653396


In [20]:
print(lora_adapter_model.peft_config)

{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', revision=None, inference_mode=False, r=8, target_modules={'k_proj', 'o_proj', 'q_proj', 'gate_proj', 'down_proj', 'up_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)}
